Vision-side tests: pure math, no camera, synthetic pixel coords.

Currently covers `src/aruco_track.py`'s `thetas_from_points`. Add more sections here as other vision pieces (homography upgrade, end-effector overlay, etc.) show up, instead of spawning a new notebook per feature.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
from math import pi, isclose
from src import aruco_track as at

Case 1: anchors level (no pixel-frame rotation vs world), left crank marker straight above its anchor -> theta1 should be pi/2.

In [2]:
pts = {
    at.L_ANCHOR_ID: np.array([0.0, 0.0]),
    at.R_ANCHOR_ID: np.array([100.0, 0.0]),
    at.L_CRANK_ID: np.array([0.0, 50.0]),   # already y-flipped (up), straight above left anchor
    at.R_CRANK_ID: np.array([100.0, 50.0]), # straight above right anchor
}
theta1, theta2 = at.thetas_from_points(pts)
assert isclose(theta1, pi / 2, abs_tol=1e-9), theta1
assert isclose(theta2, pi / 2, abs_tol=1e-9), theta2
theta1, theta2

(1.5707963267948966, 1.5707963267948966)

Case 2: whole rig rotated 30 deg in pixel space (camera not axis-aligned with the base bar). Angles should come out the same as case 1 once the anchor rotation is divided out.

In [3]:
def rot(p, a):
    c, s = np.cos(a), np.sin(a)
    return np.array([c * p[0] - s * p[1], s * p[0] + c * p[1]])

a = np.radians(30)
pts_rot = {k: rot(v, a) for k, v in pts.items()}
theta1_r, theta2_r = at.thetas_from_points(pts_rot)
assert isclose(theta1_r, theta1, abs_tol=1e-9), theta1_r
assert isclose(theta2_r, theta2, abs_tol=1e-9), theta2_r
theta1_r, theta2_r

(1.5707963267948966, 1.570796326794897)

Case 3: missing marker -> None on that side, no crash.

In [4]:
pts_missing = dict(pts)
del pts_missing[at.R_CRANK_ID]
theta1_m, theta2_m = at.thetas_from_points(pts_missing)
assert theta1_m is not None
assert theta2_m is None

pts_no_anchor = dict(pts)
del pts_no_anchor[at.L_ANCHOR_ID]
assert at.thetas_from_points(pts_no_anchor) == (None, None)

"all checks passed"

'all checks passed'